# 🔧 Cookbook 2: Creating Custom Tools

In this cookbook, you'll learn how to create your own tools for Strands Agents!

**Topics covered:**
1. The `@tool` decorator basics
2. Type hints and docstrings
3. Building practical tools
4. Advanced tool features

**Prerequisites**: Complete Cookbook 1 first!

---

In [ ]:
# Install packages if needed
!pip install strands-agents strands-agents-tools -q
print("✅ Ready to go!")

## 🎯 Why Custom Tools?

Built-in tools are great, but sometimes you need:
- Integration with your own APIs
- Custom business logic
- Domain-specific functionality

With the `@tool` decorator, you can turn any Python function into a tool!

## 📝 Step 1: Your First Custom Tool

Let's create a simple word counter tool.

In [ ]:
from strands import Agent, tool

@tool
def word_count(text: str) -> int:
    """Count the number of words in the given text.
    
    Args:
        text: The text to count words in.
    
    Returns:
        The number of words in the text.
    """
    return len(text.split())

# Create an agent with our custom tool
agent = Agent(tools=[word_count])

# Test it!
response = agent("How many words are in this sentence: 'The quick brown fox jumps over the lazy dog'?")

print("\n" + "="*50)
print("✅ Our custom tool worked!")

## 🔍 Understanding the Tool Structure

Let's break down what makes a good tool:

```python
@tool
def my_tool(param1: str, param2: int = 10) -> str:
    """Short description of what the tool does.
    
    This longer description helps the LLM understand
    when and how to use this tool.
    
    Args:
        param1: Description of param1
        param2: Description of param2 (optional)
    
    Returns:
        Description of the return value
    """
    # Your code here
    return result
```

**Key components:**
1. **Type hints** - Tell the LLM what types to use
2. **Docstring** - Helps the LLM understand the tool
3. **Args section** - Describes each parameter
4. **Return type** - What the tool returns

## 🎮 Step 2: A More Complex Tool

Let's create a tool that analyzes text sentiment (simple version).

In [ ]:
from strands import Agent, tool

@tool
def simple_sentiment(text: str) -> dict:
    """Analyze the sentiment of text using simple keyword matching.
    
    This tool looks for positive and negative words to determine
    the overall sentiment of the text.
    
    Args:
        text: The text to analyze for sentiment.
    
    Returns:
        A dictionary with 'sentiment', 'positive_count', and 'negative_count'.
    """
    positive_words = ['good', 'great', 'excellent', 'amazing', 'wonderful', 'love', 'happy', 'best', 'fantastic']
    negative_words = ['bad', 'terrible', 'awful', 'hate', 'worst', 'horrible', 'sad', 'angry', 'disappointed']
    
    text_lower = text.lower()
    
    positive_count = sum(1 for word in positive_words if word in text_lower)
    negative_count = sum(1 for word in negative_words if word in text_lower)
    
    if positive_count > negative_count:
        sentiment = "positive"
    elif negative_count > positive_count:
        sentiment = "negative"
    else:
        sentiment = "neutral"
    
    return {
        "sentiment": sentiment,
        "positive_count": positive_count,
        "negative_count": negative_count
    }

# Test it
agent = Agent(tools=[simple_sentiment])

response = agent("What's the sentiment of this review: 'This product is amazing and wonderful! I love it so much!'")

print("\n" + "="*50)
print("✅ Sentiment analysis complete!")

## 🌡️ Step 3: Tools with Multiple Parameters

Let's create a temperature converter tool with multiple parameters.

In [ ]:
from strands import Agent, tool

@tool
def convert_temperature(value: float, from_unit: str, to_unit: str) -> float:
    """Convert temperature between Celsius, Fahrenheit, and Kelvin.
    
    Args:
        value: The temperature value to convert.
        from_unit: The source unit ('celsius', 'fahrenheit', or 'kelvin').
        to_unit: The target unit ('celsius', 'fahrenheit', or 'kelvin').
    
    Returns:
        The converted temperature value.
    """
    # Convert to Celsius first
    if from_unit.lower() == 'fahrenheit':
        celsius = (value - 32) * 5/9
    elif from_unit.lower() == 'kelvin':
        celsius = value - 273.15
    else:
        celsius = value
    
    # Convert from Celsius to target
    if to_unit.lower() == 'fahrenheit':
        return round(celsius * 9/5 + 32, 2)
    elif to_unit.lower() == 'kelvin':
        return round(celsius + 273.15, 2)
    else:
        return round(celsius, 2)

agent = Agent(tools=[convert_temperature])

response = agent("Convert 100 degrees Fahrenheit to Celsius.")

print("\n" + "="*50)

## 🔄 Step 4: Tools with Default Parameters

Default parameters make tools more flexible.

In [ ]:
from strands import Agent, tool

@tool
def generate_greeting(name: str, language: str = "english", formal: bool = False) -> str:
    """Generate a personalized greeting in different languages.
    
    Args:
        name: The name of the person to greet.
        language: The language for the greeting (english, spanish, french, german).
        formal: Whether to use formal language.
    
    Returns:
        A personalized greeting string.
    """
    greetings = {
        "english": {"informal": f"Hey {name}!", "formal": f"Good day, {name}."},
        "spanish": {"informal": f"¡Hola {name}!", "formal": f"Buenos días, {name}."},
        "french": {"informal": f"Salut {name}!", "formal": f"Bonjour, {name}."},
        "german": {"informal": f"Hallo {name}!", "formal": f"Guten Tag, {name}."}
    }
    
    lang = language.lower()
    style = "formal" if formal else "informal"
    
    if lang in greetings:
        return greetings[lang][style]
    return f"Hello, {name}!"

agent = Agent(tools=[generate_greeting])

# Try different requests
print("Request 1:")
response = agent("Greet Maria in Spanish formally.")

print("\n\nRequest 2:")
response = agent("Say hi to John in a casual way.")

## 📊 Step 5: Tools that Return Structured Data

Tools can return complex data structures.

In [ ]:
from strands import Agent, tool
import random

@tool
def get_stock_info(symbol: str) -> dict:
    """Get stock information for a given ticker symbol.
    
    Note: This returns simulated data for demonstration purposes.
    
    Args:
        symbol: The stock ticker symbol (e.g., 'AAPL', 'GOOGL').
    
    Returns:
        A dictionary containing stock information including price,
        change, volume, and market cap.
    """
    # Simulated stock data (in real use, you'd call an actual API)
    simulated_stocks = {
        "AAPL": {"name": "Apple Inc.", "base_price": 175.50},
        "GOOGL": {"name": "Alphabet Inc.", "base_price": 140.25},
        "MSFT": {"name": "Microsoft Corp.", "base_price": 380.00},
        "AMZN": {"name": "Amazon.com Inc.", "base_price": 178.50},
    }
    
    symbol = symbol.upper()
    
    if symbol not in simulated_stocks:
        return {"error": f"Stock symbol {symbol} not found"}
    
    stock = simulated_stocks[symbol]
    # Add some randomness for realism
    price = stock["base_price"] * (1 + random.uniform(-0.05, 0.05))
    change = random.uniform(-3, 3)
    
    return {
        "symbol": symbol,
        "name": stock["name"],
        "price": round(price, 2),
        "change_percent": round(change, 2),
        "volume": random.randint(10_000_000, 50_000_000),
        "market_cap": f"${random.randint(500, 3000)}B"
    }

agent = Agent(tools=[get_stock_info])

response = agent("What's the current stock information for Apple (AAPL) and Microsoft (MSFT)?")

## 🎭 Step 6: Combining Custom and Built-in Tools

The real power comes from combining tools!

In [ ]:
from strands import Agent, tool
from strands_tools import calculator, current_time

@tool
def calculate_tip(bill_amount: float, service_quality: str = "good") -> dict:
    """Calculate the tip amount based on bill and service quality.
    
    Args:
        bill_amount: The total bill amount in dollars.
        service_quality: Quality of service ('poor', 'average', 'good', 'excellent').
    
    Returns:
        Dictionary with tip percentage, tip amount, and total.
    """
    tip_rates = {
        "poor": 0.10,
        "average": 0.15,
        "good": 0.18,
        "excellent": 0.25
    }
    
    rate = tip_rates.get(service_quality.lower(), 0.15)
    tip_amount = bill_amount * rate
    
    return {
        "bill_amount": bill_amount,
        "tip_percentage": f"{rate * 100}%",
        "tip_amount": round(tip_amount, 2),
        "total": round(bill_amount + tip_amount, 2)
    }

@tool
def split_bill(total: float, num_people: int) -> dict:
    """Split a bill among multiple people.
    
    Args:
        total: The total amount to split.
        num_people: Number of people splitting the bill.
    
    Returns:
        Dictionary with amount per person and remainder.
    """
    per_person = total / num_people
    
    return {
        "total": total,
        "num_people": num_people,
        "amount_per_person": round(per_person, 2)
    }

# Create an agent with both custom and built-in tools
agent = Agent(tools=[calculate_tip, split_bill, calculator, current_time])

response = agent("""
I have a restaurant bill of $85.50 and the service was excellent.
Please calculate the tip, then split the total among 4 people.
Also, what time is it now?
""")

print("\n" + "="*50)
print("✅ Multiple tools working together!")

## 🏋️ Exercise: Build Your Own Tool

Now it's your turn! Create a custom tool that:
- Takes at least 2 parameters
- Has a clear docstring
- Returns useful information

Ideas:
- A unit converter (miles to km, etc.)
- A password strength checker
- A text summarizer (count sentences, average word length, etc.)

In [ ]:
from strands import Agent, tool

# 🏋️ YOUR EXERCISE: Create your own tool here!
@tool
def my_custom_tool(param1: str, param2: int = 5) -> dict:
    """Description of what your tool does.
    
    Args:
        param1: Description of param1.
        param2: Description of param2.
    
    Returns:
        What your tool returns.
    """
    # Your code here
    return {"result": "Your result here"}

# Test your tool
agent = Agent(tools=[my_custom_tool])
response = agent("Test your tool with an appropriate question!")

## ✅ Summary

Congratulations! You've learned:

1. **Using `@tool` decorator** - Turn any function into a tool
2. **Writing good docstrings** - Help the LLM understand your tool
3. **Type hints** - Tell the LLM what types to use
4. **Default parameters** - Make tools more flexible
5. **Returning structured data** - Complex responses
6. **Combining tools** - Mix custom and built-in tools

---

## 🚀 Next Steps

Continue to **Cookbook 3: Model Providers** to learn about different AI models!

---

## 📚 Resources

- [Custom Tools Documentation](https://strandsagents.com/latest/user-guide/concepts/tools/custom-tools/)
- [Built-in Tools Reference](https://github.com/strands-agents/tools)